# Advanced Machine Learning (21CSE425T) — Unit 4: Association Rule Mining

Standalone Colab notebook for **T10–T12**: Market Basket Analysis, Apriori from scratch, and FP-Growth for classification. Run top-to-bottom; the setup cell installs all dependencies.

---
# Unit 4 — Association Rule Mining

Installs mlxtend + pandas, then covers **Market Basket Analysis** (T10), the **Apriori algorithm** both
from scratch and via mlxtend (T11), and **FP-Growth used for classification** (T12).

### Unit 4 — Setup & installs

In [ ]:
!pip -q install mlxtend pandas 2>/dev/null
import pandas as pd, mlxtend
print("pandas :", pd.__version__)
print("mlxtend:", mlxtend.__version__)

## T10 — Association Rule Mining in a real-time application (Market Basket Analysis)

A retail **market-basket** example: one-hot encode transactions with `TransactionEncoder`, mine frequent
itemsets with **Apriori**, and derive **association rules** ranked by *support*, *confidence* and *lift* —
the engine behind "customers who bought X also bought Y" recommendations.

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import pandas as pd

# ---- Shopping transactions (swap for a real POS log) ----
transactions = [
    ["milk", "bread", "butter"],
    ["bread", "butter"],
    ["milk", "bread"],
    ["milk", "bread", "butter", "eggs"],
    ["bread", "eggs"],
    ["milk", "butter", "eggs"],
    ["milk", "bread", "butter"],
    ["bread", "butter", "eggs"],
]
te = TransactionEncoder()
df = pd.DataFrame(te.fit_transform(transactions), columns=te.columns_)
print("One-hot encoded basket matrix:"); print(df.astype(int), "\n")

freq = apriori(df, min_support=0.3, use_colnames=True)
print("Frequent itemsets (min_support=0.3):")
print(freq.sort_values("support", ascending=False), "\n")

rules = association_rules(freq, metric="confidence", min_threshold=0.6)
rules = rules.sort_values("lift", ascending=False)
cols = ["antecedents", "consequents", "support", "confidence", "lift"]
print("Association rules (confidence >= 0.6), sorted by lift:")
print(rules[cols].to_string(index=False))

## T11 — The Apriori algorithm (from scratch) + validation

We implement Apriori directly — iterative candidate generation, support counting and the downward-closure
pruning — to extract **large itemsets** and **rules**, then confirm the frequent itemsets match mlxtend's.

In [ ]:
from itertools import combinations
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori as mlx_apriori

dataset = [
    ["milk", "bread", "butter"], ["bread", "butter"], ["milk", "bread"],
    ["milk", "bread", "butter", "eggs"], ["bread", "eggs"],
    ["milk", "butter", "eggs"], ["milk", "bread", "butter"], ["bread", "butter", "eggs"],
]

def apriori_from_scratch(transactions, min_support=0.3):
    T = [set(t) for t in transactions]
    n = len(T)
    def support(itemset):
        return sum(1 for t in T if itemset <= t) / n

    items = sorted({i for t in T for i in t})
    # L1: frequent 1-itemsets
    current = [frozenset([i]) for i in items if support(frozenset([i])) >= min_support]
    all_frequent = {}
    k = 1
    while current:
        for its in current:
            all_frequent[its] = support(its)
        # candidate generation: join size-k frequent sets into size-(k+1) candidates
        candidates = set()
        for a, b in combinations(current, 2):
            union = a | b
            if len(union) == k + 1:
                # prune: every k-subset of a candidate must itself be frequent
                if all(frozenset(s) in all_frequent for s in combinations(union, k)):
                    candidates.add(union)
        current = [c for c in candidates if support(c) >= min_support]
        k += 1
    return all_frequent

def generate_rules(frequent, min_conf=0.6):
    rules = []
    for itemset, sup in frequent.items():
        if len(itemset) < 2:
            continue
        for r in range(1, len(itemset)):
            for ante in combinations(itemset, r):
                ante = frozenset(ante); cons = itemset - ante
                conf = sup / frequent[ante]
                if conf >= min_conf:
                    lift = conf / frequent[cons]
                    rules.append((set(ante), set(cons), round(sup,3), round(conf,3), round(lift,3)))
    return rules

frequent = apriori_from_scratch(dataset, min_support=0.3)
print("Large (frequent) itemsets from scratch:")
for its, sup in sorted(frequent.items(), key=lambda kv: (len(kv[0]), -kv[1])):
    print(f"  support={sup:.3f}  {set(its)}")

print("\nAssociation rules (confidence >= 0.6):")
for ante, cons, sup, conf, lift in generate_rules(frequent, 0.6):
    print(f"  {ante} -> {cons} | sup={sup} conf={conf} lift={lift}")

# ---- Validate against mlxtend ----
te = TransactionEncoder(); dfm = pd.DataFrame(te.fit_transform(dataset), columns=te.columns_)
mlx = mlx_apriori(dfm, min_support=0.3, use_colnames=True)
mlx_sets = {frozenset(s) for s in mlx["itemsets"]}
print("\nFrequent itemsets match mlxtend:", mlx_sets == set(frequent.keys()))

## T12 — Building a Frequent Pattern (FP) Tree for classification

**FP-Growth** mines frequent patterns without candidate generation (via the compact FP-tree). Here we use
those patterns as **engineered binary features** for a classifier: we discretise Iris into items, mine
frequent patterns with `fpgrowth`, turn each pattern into a feature ("does this sample contain the
pattern?"), and show the pattern features train a strong classifier — i.e. association analysis feeding
classification.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth
import numpy as np, pandas as pd

# ---- 1) Discretise each numeric feature into low/med/high "items" ----
iris = load_iris()
X, y = iris.data, iris.target
feat_items = []
for j, name in enumerate(iris.feature_names):
    bins = np.quantile(X[:, j], [0, 0.33, 0.66, 1.0])
    labels = [f"{name.split(' (')[0]}={lvl}" for lvl in ["low", "med", "high"]]
    cats = pd.cut(X[:, j], bins=bins, labels=labels, include_lowest=True)
    feat_items.append(cats.astype(str))
transactions = [list(row) for row in zip(*feat_items)]   # one item-set per flower

te = TransactionEncoder()
df_items = pd.DataFrame(te.fit_transform(transactions), columns=te.columns_)

# ---- 2) Mine frequent patterns with FP-Growth ----
patterns = fpgrowth(df_items, min_support=0.15, use_colnames=True)
patterns = patterns[patterns["itemsets"].apply(len) >= 2].reset_index(drop=True)
print(f"Mined {len(patterns)} frequent patterns (size>=2). Examples:")
for s in patterns["itemsets"].head(6):
    print("  ", set(s))

# ---- 3) Turn each pattern into a binary feature: sample contains the whole pattern? ----
pattern_sets = [set(s) for s in patterns["itemsets"]]
def to_pattern_features(item_df):
    present = item_df.values
    cols = item_df.columns.tolist()
    feats = np.zeros((len(item_df), len(pattern_sets)), dtype=int)
    for i in range(len(item_df)):
        owned = {cols[j] for j in range(len(cols)) if present[i, j]}
        for k, ps in enumerate(pattern_sets):
            feats[i, k] = int(ps <= owned)
    return feats

Xp = to_pattern_features(df_items)
Xtr, Xte, ytr, yte = train_test_split(Xp, y, test_size=0.3, random_state=0, stratify=y)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print(f"\nClassification accuracy using FP-pattern features: "
      f"{accuracy_score(yte, clf.predict(Xte)):.3f}")